# Statistical Engine

1. Ingest a CSV with control/variant metrics
2. Calculate p-values, confidence intervals, relative lift
3. Flag underpowered tests (sample size checks)
4. Output a structured result dictionary

In [1]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import math

---
### 1. Ingest a CSV with control/variant metrics

In [2]:
df = pd.read_csv('../data/ab_test_experiments_v2.csv')
df.head(3)

,test_id,test_name,product_area,hypothesis,primary_metric,secondary_metric,device_type,audience_segment,start_date,end_date,...,relative_lift_pct,secondary_metric_control,secondary_metric_variant,avg_order_value,control_revenue,variant_revenue,revenue_delta,outcome_type,analyst,notes
0,EXP-1000,Category Filter UX v1,Acquisition,Clearer value proposition will improve activation,signup_rate,bounce_rate,Mobile,Returning Users,2023-10-15,2023-10-23,...,7.46,0.4066,0.3666,407.76,92969.28,54639.84,-38329.44,underpowered,Sam T.,Test targeting returning users on mobile. Hypo...
1,EXP-1001,Trust Badge Placement v2,Onboarding,Reducing friction will increase conversion,signup_rate,avg_session_duration,Desktop,Free Tier,2023-05-31,2023-07-12,...,14.69,234.1898,268.4530,431.28,5879640.24,2889144.72,-2990495.52,clear_winner,Sam T.,Test targeting free tier on desktop. Hypothesi...
2,EXP-1002,Progress Bar Addition v3,Onboarding,Urgency messaging will drive faster decisions,signup_rate,pages_per_session,Mobile,Premium Members,2023-04-23,2023-06-04,...,12.14,0.4045,0.4046,56.44,3447129.44,1656683.32,-1790446.12,clear_winner,Alex R.,Test targeting premium members on mobile. Hypo...


In [3]:
# Pre-test planning columns (inputs known before the test runs)
pre_test_data = df[['test_id', 'test_name', 'product_area', 'hypothesis', 'primary_metric',
       'secondary_metric', 'device_type', 'audience_segment', 'start_date',
       'end_date', 'test_duration_days', 'traffic_split_control_pct',
       'test_direction', 'test_tails', 'alpha', 'mde_relative_pct',
       'control_visitors', 'variant_visitors', 'total_visitors',
       'control_conversions', 'variant_conversions', 'relative_lift_pct',
       'secondary_metric_control', 'secondary_metric_variant',
       'avg_order_value', 'control_revenue', 'variant_revenue',
       'revenue_delta', 'analyst', 'notes']]

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 33 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   test_id                    200 non-null    object 
 1   test_name                  200 non-null    object 
 2   product_area               200 non-null    object 
 3   hypothesis                 200 non-null    object 
 4   primary_metric             200 non-null    object 
 5   secondary_metric           200 non-null    object 
 6   device_type                200 non-null    object 
 7   audience_segment           200 non-null    object 
 8   start_date                 200 non-null    object 
 9   end_date                   200 non-null    object 
 10  test_duration_days         200 non-null    int64  
 11  traffic_split_control_pct  200 non-null    int64  
 12  test_direction             200 non-null    object 
 13  test_tails                 200 non-null    object 

In [5]:
# Outcome labels in dataset — we'll reproduce these from first principles
df.outcome_type.value_counts()

clear_winner      45
underpowered      39
neutral           37
clear_loser       33
false_positive    25
novelty_effect    21
Name: outcome_type, dtype: int64

---
### 2a. P-Values — Two-proportion z-test with pooled variance

**Formula:**
$$z = \frac{\hat{p}_v - \hat{p}_c}{\sqrt{\hat{p}(1-\hat{p})\left(\frac{1}{n_c} + \frac{1}{n_v}\right)}}$$

where $\hat{p}$ is the pooled proportion across both groups.

- **Two-tailed:** $p = 2 \cdot P(Z > |z|)$ — detects any difference
- **One-tailed (superiority / non-inferiority):** $p = P(Z > z)$ — tests if variant is better

In [6]:
def calculate_pvalue(
    control_visitors: int,
    variant_visitors: int,
    control_conversions: int,
    variant_conversions: int,
    test_tails: str = 'two_tailed',
    test_direction: str = 'superiority',
) -> float:
    """
    Two-proportion z-test with pooled variance.

    Parameters
    ----------
    control_visitors, variant_visitors   : sample sizes
    control_conversions, variant_conversions : successes
    test_tails      : 'one_tailed' | 'two_tailed'
    test_direction  : 'superiority' | 'non_inferiority'

    Returns
    -------
    float : p-value rounded to 4 decimal places
    """
    p_c = control_conversions / control_visitors
    p_v = variant_conversions / variant_visitors

    # Pooled proportion and standard error under H0
    p_pool = (control_conversions + variant_conversions) / (control_visitors + variant_visitors)
    se = math.sqrt(p_pool * (1 - p_pool) * (1 / control_visitors + 1 / variant_visitors))

    z_stat = (p_v - p_c) / se

    if test_tails == 'two_tailed':
        p_value = 2 * (1 - stats.norm.cdf(abs(z_stat)))
    else:
        # superiority and non_inferiority both test right tail (variant >= control)
        if test_direction in ('superiority', 'non_inferiority'):
            p_value = 1 - stats.norm.cdf(z_stat)
        else:
            p_value = stats.norm.cdf(z_stat)

    return round(p_value, 4)

In [7]:
# Smoke tests
print("Underpowered (expect ~0.23):", calculate_pvalue(1406, 769, 228, 134, 'one_tailed', 'superiority'))
print("Clear winner  (expect ~0.00):", calculate_pvalue(215388, 92310, 13633, 6699, 'one_tailed', 'superiority'))

Underpowered (expect ~0.23): 0.2346
Clear winner  (expect ~0.00): 0.0


In [8]:
def add_pvalues(df: pd.DataFrame) -> pd.DataFrame:
    """Apply calculate_pvalue() row-wise and add p_values + Significant_Results columns."""
    results = []
    for _, row in df.iterrows():
        res = calculate_pvalue(
            control_visitors=int(row['control_visitors']),
            variant_visitors=int(row['variant_visitors']),
            control_conversions=int(row['control_conversions']),
            variant_conversions=int(row['variant_conversions']),
            test_tails=row['test_tails'],
            test_direction=row['test_direction'],
        )
        results.append(res)

    out = df.copy()
    out['p_values'] = results
    out['Significant_Results'] = np.where(out['p_values'] < out['alpha'], 'Significant', 'Not-Significant')
    return out

test_df = add_pvalues(df)
test_df[['test_id', 'test_tails', 'test_direction', 'alpha', 'p_values', 'Significant_Results']].head(10)

,test_id,test_tails,test_direction,alpha,p_values,Significant_Results
0,EXP-1000,one_tailed,superiority,0.01,0.2346,Not-Significant
1,EXP-1001,one_tailed,superiority,0.05,0.0000,Significant
2,EXP-1002,one_tailed,superiority,0.05,0.0000,Significant
3,EXP-1003,two_tailed,superiority,0.05,0.0000,Significant
4,EXP-1004,two_tailed,superiority,0.05,0.9160,Not-Significant
5,EXP-1005,one_tailed,superiority,0.01,0.5618,Not-Significant
6,EXP-1006,two_tailed,superiority,0.05,0.0000,Significant
7,EXP-1007,one_tailed,non_inferiority,0.05,1.0000,Not-Significant
8,EXP-1008,two_tailed,superiority,0.05,0.0000,Significant
9,EXP-1009,one_tailed,non_inferiority,0.05,1.0000,Not-Significant


In [9]:
print(f"Significant: {(test_df.Significant_Results == 'Significant').sum()} / {len(test_df)}")
test_df[['alpha', 'p_values', 'Significant_Results']].value_counts()

Significant: 87 / 200


alpha  p_values  Significant_Results
0.05   0.0000    Significant            58
       1.0000    Not-Significant        12
0.01   0.0000    Significant            10
0.05   0.3253    Not-Significant         2
       0.6168    Not-Significant         2
                                        ..
       0.1019    Not-Significant         1
       0.0726    Not-Significant         1
       0.0679    Not-Significant         1
       0.0645    Not-Significant         1
       0.3405    Not-Significant         1
Length: 120, dtype: int64

---
### 2b. Confidence Intervals

CI is built on **unpooled** SE (actual group variance, not H0 assumption):

$$SE = \sqrt{\frac{\hat{p}_c(1-\hat{p}_c)}{n_c} + \frac{\hat{p}_v(1-\hat{p}_v)}{n_v}}$$

- **Two-tailed:** $\text{lift} \pm z_{\alpha/2} \cdot SE$ → symmetric interval
- **One-tailed:** $\text{lift} - z_{\alpha} \cdot SE$ → lower bound only (one-sided guarantee)

In [10]:
def confidence_interval(
    control_conversions: int,
    control_visitors: int,
    variant_conversions: int,
    variant_visitors: int,
    alpha: float,
    test_tails: str = 'two_tailed',
) -> dict:
    """
    Confidence interval for the absolute lift (p_variant - p_control).

    Two-tailed → symmetric (ci_lower, ci_upper)
    One-tailed → lower bound only (ci_lower, ci_upper=None)

    Returns
    -------
    dict: absolute_lift, ci_lower, ci_upper, z_critical
    """
    p_c = control_conversions / control_visitors
    p_v = variant_conversions / variant_visitors
    lift = p_v - p_c

    # Unpooled SE — reflects actual variance, not H0 pooled estimate
    se = math.sqrt(
        p_c * (1 - p_c) / control_visitors +
        p_v * (1 - p_v) / variant_visitors
    )

    if test_tails == 'two_tailed':
        z = stats.norm.ppf(1 - alpha / 2)       # e.g. 1.96 at α=0.05
        ci_lower = round(lift - z * se, 6)
        ci_upper = round(lift + z * se, 6)
    else:
        z = stats.norm.ppf(1 - alpha)            # e.g. 1.645 at α=0.05
        ci_lower = round(lift - z * se, 6)       # one-sided lower bound
        ci_upper = None

    return {
        'absolute_lift': round(lift, 6),
        'ci_lower': ci_lower,
        'ci_upper': ci_upper,
        'z_critical': round(z, 4),
    }

In [11]:
# Smoke tests
print("One-tailed (underpowered):", confidence_interval(228, 1406, 134, 769, 0.01, 'one_tailed'))
print("Two-tailed (clear winner):", confidence_interval(13633, 215388, 6699, 92310, 0.05, 'two_tailed'))

One-tailed (underpowered): {'absolute_lift': 0.01209, 'ci_lower': -0.027097, 'ci_upper': None, 'z_critical': 2.3263}
Two-tailed (clear winner): {'absolute_lift': 0.009276, 'ci_lower': 0.007311, 'ci_upper': 0.01124, 'z_critical': 1.96}


---
### 3. Flag Underpowered Tests — Sample Size & Observed Power

**Required Sample Size Formula:**

$$n = \frac{\left(z_\alpha\sqrt{2\bar{p}(1-\bar{p})} + z_\beta\sqrt{p_c(1-p_c) + p_v(1-p_v)}\right)^2}{(p_v - p_c)^2}$$

where $\bar{p} = (p_c + p_v)/2$, $z_\alpha$ depends on tail type, $z_\beta = \Phi^{-1}(\text{power})$.

**Observed Power:** how much power the test *actually had* given real sample sizes.

In [12]:
def required_sample_size(
    baseline_rate: float,
    mde_relative_pct: float,
    alpha: float,
    power: float = 0.80,
    tails: str = 'two_tailed',
) -> dict:
    """
    Calculate required sample size per variant using the two-proportion
    z-test power formula.

    Parameters
    ----------
    baseline_rate    : control conversion rate (e.g. 0.10 for 10%)
    mde_relative_pct : minimum detectable effect as relative % (e.g. 5.0 for 5%)
    alpha            : significance level (Type I error)
    power            : desired power = 1 − β (default 0.80)
    tails            : 'one_tailed' | 'two_tailed'

    Returns
    -------
    dict with sample sizes, rates, and z-scores
    """
    absolute_mde  = baseline_rate * (mde_relative_pct / 100)
    variant_rate  = baseline_rate + absolute_mde
    p_avg         = (baseline_rate + variant_rate) / 2

    # Critical z-scores
    z_alpha = stats.norm.ppf(1 - alpha) if tails == 'one_tailed' else stats.norm.ppf(1 - alpha / 2)
    z_beta  = stats.norm.ppf(power)

    # Two-proportion sample size formula
    numerator   = (z_alpha * math.sqrt(2 * p_avg * (1 - p_avg)) +
                   z_beta  * math.sqrt(baseline_rate * (1 - baseline_rate) +
                                       variant_rate  * (1 - variant_rate))) ** 2
    denominator = absolute_mde ** 2
    n = math.ceil(numerator / denominator)

    return {
        'sample_size_per_variant': n,
        'total_sample_size': n * 2,
        'variant_rate_assumed': round(variant_rate, 6),
        'absolute_mde': round(absolute_mde, 6),
        'z_alpha': round(z_alpha, 4),
        'z_beta': round(z_beta, 4),
    }

In [13]:
def observed_power(
    control_conversions: int,
    control_visitors: int,
    variant_conversions: int,
    variant_visitors: int,
    alpha: float,
    test_tails: str = 'two_tailed',
) -> float:
    """
    Estimate the statistical power the test actually had, given real sample sizes.
    Uses the non-centrality approach: how many SEs the true effect sits above
    the critical threshold.

    Returns
    -------
    float : power between 0 and 1
    """
    p_c = control_conversions / control_visitors
    p_v = variant_conversions / variant_visitors
    absolute_lift = abs(p_v - p_c)

    # SE under alternative hypothesis (unpooled)
    se_alt = math.sqrt(
        p_c * (1 - p_c) / control_visitors +
        p_v * (1 - p_v) / variant_visitors
    )

    if se_alt == 0:
        return 0.0

    z_crit = stats.norm.ppf(1 - alpha / 2) if test_tails == 'two_tailed' else stats.norm.ppf(1 - alpha)

    # Non-centrality parameter: true effect in units of SE
    ncp   = absolute_lift / se_alt
    power = 1 - stats.norm.cdf(z_crit - ncp)

    return round(float(power), 4)

In [14]:
def flag_underpowered(
    control_conversions: int,
    control_visitors: int,
    variant_conversions: int,
    variant_visitors: int,
    baseline_rate: float,
    mde_relative_pct: float,
    alpha: float,
    test_tails: str,
    power_threshold: float = 0.80,
) -> dict:
    """
    Compare required vs actual sample sizes and compute observed power
    to determine if a test was underpowered.

    Returns
    -------
    dict with power diagnostics and underpowered flag
    """
    req = required_sample_size(baseline_rate, mde_relative_pct, alpha, power_threshold, test_tails)
    obs = observed_power(control_conversions, control_visitors,
                         variant_conversions, variant_visitors,
                         alpha, test_tails)

    actual_total   = control_visitors + variant_visitors
    required_total = req['total_sample_size']

    return {
        'observed_power':              obs,
        'required_sample_per_variant': req['sample_size_per_variant'],
        'required_sample_total':       required_total,
        'actual_sample_total':         actual_total,
        'sample_deficit':              max(0, required_total - actual_total),
        'is_underpowered':             obs < power_threshold,
        'power_threshold':             power_threshold,
    }

In [15]:
# Smoke tests
row0 = df.iloc[0]
print("Required sample size (EXP-1000):",
      required_sample_size(row0.control_conversion_rate, row0.mde_relative_pct, row0.alpha, 0.80, row0.test_tails))

print("\nObserved power (EXP-1000):",
      observed_power(int(row0.control_conversions), int(row0.control_visitors),
                     int(row0.variant_conversions), int(row0.variant_visitors),
                     row0.alpha, row0.test_tails))

print("\nUnderpowered flag (EXP-1000):",
      flag_underpowered(int(row0.control_conversions), int(row0.control_visitors),
                        int(row0.variant_conversions), int(row0.variant_visitors),
                        row0.control_conversion_rate, row0.mde_relative_pct,
                        row0.alpha, row0.test_tails))

Required sample size (EXP-1000): {'sample_size_per_variant': 81153, 'total_sample_size': 162306, 'variant_rate_assumed': 0.168039, 'absolute_mde': 0.005839, 'z_alpha': 2.3263, 'z_beta': 0.8416}

Observed power (EXP-1000): 0.0539

Underpowered flag (EXP-1000): {'observed_power': 0.0539, 'required_sample_per_variant': 81153, 'required_sample_total': 162306, 'actual_sample_total': 2175, 'sample_deficit': 160131, 'is_underpowered': True, 'power_threshold': 0.8}


---
### 4. Structured Result Dictionary — `analyze_experiment()`

Assembles all outputs (p-value, CI, lift, power diagnostics) into a single
structured dict per experiment. This is the contract that Phase 3 (LLM narrative)
will consume.

In [16]:
def analyze_experiment(row: pd.Series, power_threshold: float = 0.80) -> dict:
    """
    Full statistical analysis for a single A/B experiment row.

    Parameters
    ----------
    row             : a single row from the experiments DataFrame
    power_threshold : minimum acceptable power (default 0.80)

    Returns
    -------
    Structured dict — the output contract for Phase 3 LLM narrative.
    """
    c_conv = int(row['control_conversions'])
    c_vis  = int(row['control_visitors'])
    v_conv = int(row['variant_conversions'])
    v_vis  = int(row['variant_visitors'])

    p_val = calculate_pvalue(c_vis, v_vis, c_conv, v_conv,
                              row['test_tails'], row['test_direction'])

    ci    = confidence_interval(c_conv, c_vis, v_conv, v_vis,
                                row['alpha'], row['test_tails'])

    power = flag_underpowered(c_conv, c_vis, v_conv, v_vis,
                              row['control_conversion_rate'],
                              row['mde_relative_pct'],
                              row['alpha'], row['test_tails'],
                              power_threshold)

    is_sig = p_val < row['alpha']
    p_c    = row['control_conversion_rate']
    rel_lift = round((ci['absolute_lift'] / p_c) * 100, 2) if p_c > 0 else None

    return {
        # Identity
        'test_id':              row['test_id'],
        'test_name':            row['test_name'],
        'product_area':         row['product_area'],
        'hypothesis':           row['hypothesis'],
        'primary_metric':       row['primary_metric'],
        'audience_segment':     row['audience_segment'],
        'test_duration_days':   row['test_duration_days'],

        # Test design
        'test_tails':           row['test_tails'],
        'test_direction':       row['test_direction'],
        'alpha':                row['alpha'],
        'mde_relative_pct':     row['mde_relative_pct'],

        # Sample sizes
        'control_visitors':     c_vis,
        'variant_visitors':     v_vis,
        'control_conversions':  c_conv,
        'variant_conversions':  v_conv,

        # Core rates
        'control_rate':         round(row['control_conversion_rate'], 6),
        'variant_rate':         round(row['variant_conversion_rate'], 6),

        # Statistical output
        'p_value':              p_val,
        'is_significant':       is_sig,
        'absolute_lift':        ci['absolute_lift'],
        'relative_lift_pct':    rel_lift,
        'ci_lower':             ci['ci_lower'],
        'ci_upper':             ci['ci_upper'],

        # Power diagnostics
        'observed_power':              power['observed_power'],
        'is_underpowered':             power['is_underpowered'],
        'required_sample_per_variant': power['required_sample_per_variant'],
        'required_sample_total':       power['required_sample_total'],
        'actual_sample_total':         power['actual_sample_total'],
        'sample_deficit':              power['sample_deficit'],

        # Revenue context
        'revenue_delta':        row['revenue_delta'],
        'avg_order_value':      row['avg_order_value'],

        # Ground truth (for validation only — not fed to LLM)
        'outcome_type':         row['outcome_type'],
        'analyst':              row['analyst'],
    }

In [17]:
# Single experiment result
result = analyze_experiment(df.iloc[0])
for k, v in result.items():
    print(f"  {k:<35} {v}")

  test_id                             EXP-1000
  test_name                           Category Filter UX v1
  product_area                        Acquisition
  hypothesis                          Clearer value proposition will improve activation
  primary_metric                      signup_rate
  audience_segment                    Returning Users
  test_duration_days                  8
  test_tails                          one_tailed
  test_direction                      superiority
  alpha                               0.01
  mde_relative_pct                    3.6
  control_visitors                    1406
  variant_visitors                    769
  control_conversions                 228
  variant_conversions                 134
  control_rate                        0.1622
  variant_rate                        0.1743
  p_value                             0.2346
  is_significant                      False
  absolute_lift                       0.01209
  relative_lift_pct              

In [18]:
# Run all 200 experiments
results = [analyze_experiment(row) for _, row in df.iterrows()]
results_df = pd.DataFrame(results)

print(f"Total experiments : {len(results_df)}")
print(f"Significant       : {results_df.is_significant.sum()}")
print(f"Underpowered      : {results_df.is_underpowered.sum()}")
print(f"Avg observed power: {results_df.observed_power.mean():.2%}")
print()
results_df[['test_id','p_value','is_significant','relative_lift_pct',
            'observed_power','is_underpowered','sample_deficit','outcome_type']].head(10)

Total experiments : 200
Significant       : 87
Underpowered      : 109
Avg observed power: 57.31%



,test_id,p_value,is_significant,relative_lift_pct,observed_power,is_underpowered,sample_deficit,outcome_type
0,EXP-1000,0.2346,False,7.45,0.0539,True,160131,underpowered
1,EXP-1001,0.0000,True,14.65,1.0000,False,0,clear_winner
2,EXP-1002,0.0000,True,12.14,1.0000,False,0,clear_winner
3,EXP-1003,0.0000,True,13.45,1.0000,False,0,clear_winner
4,EXP-1004,0.9160,False,-0.08,0.0318,True,0,false_positive
5,EXP-1005,0.5618,False,-2.91,0.0149,True,477859,underpowered
6,EXP-1006,0.0000,True,5.58,0.9978,False,6854,clear_winner
7,EXP-1007,1.0000,False,-15.98,1.0000,False,0,clear_loser
8,EXP-1008,0.0000,True,8.16,1.0000,False,0,novelty_effect
9,EXP-1009,1.0000,False,-8.48,1.0000,False,0,clear_loser


---
### Validation — Engine vs Recorded Outcomes

In [19]:
# How well do our computed flags match the human-labelled outcome_type?
results_df['outcome_positive'] = results_df['outcome_type'].isin(['significant_positive', 'clear_winner', 'novelty_effect'])

sig_match       = (results_df['is_significant']  == results_df['outcome_positive']).mean()
underpow_match  = (results_df['is_underpowered'] == (results_df['outcome_type'] == 'underpowered')).mean()

print(f"Significance agreement with outcome_type : {sig_match:.1%}")
print(f"Underpowered flag agreement              : {underpow_match:.1%}")

print("\nOutcome breakdown:")
print(results_df.groupby(['outcome_type', 'is_significant', 'is_underpowered']).size().to_string())

Significance agreement with outcome_type : 83.5%
Underpowered flag agreement              : 64.0%

Outcome breakdown:
outcome_type    is_significant  is_underpowered
clear_loser     False           False              15
                                True                3
                True            False              14
                                True                1
clear_winner    False           True                5
                True            False              40
false_positive  False           True               23
                True            True                2
neutral         False           True               31
                True            False               3
                                True                3
novelty_effect  False           True                1
                True            False              18
                                True                2
underpowered    False           True               35
                True    